In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt

# 1. 데이터 로드
df = pd.read_csv('data/KBO_Season_Master_Data.csv')

# 2. 분석 지표 설정 (매핑을 통해 출력 이름 변경)
feature_mapping = {
    'WAR_total': 'WAR_total',
    'ERA': 'ERA (평균자책점)',
    'OPS': 'OPS (출루+장타율)',
    '득점': '득점 (R)',
    'WHIP': 'WHIP',
    'F%': 'F% (수비율)',
    'SB%': 'SB% (도루성공률)'
}

features = list(feature_mapping.keys())
target = '승률'

results = []

# 3. 스피어먼 상관계수 및 p-value 계산
for col in features:
    # 데이터 내 결측치가 있을 경우 제거하고 계산
    valid_data = df[[target, col]].dropna()
    rho, p_val = stats.spearmanr(valid_data[target], valid_data[col])
    
    results.append({
        '분석 지표': feature_mapping[col],
        'Spearman_Rho': round(rho, 3),
        'P_value': round(p_val, 3),
        '통계적 유의성(0.05미만)': '유의함' if p_val < 0.05 else '유의하지 않음'
    })

# 결과 데이터프레임 생성 및 출력
analysis_results = pd.DataFrame(results)

# 상관계수 절대값 기준으로 정렬하여 중요도 파악
analysis_results['abs_rho'] = analysis_results['Spearman_Rho'].abs()
sorted_results = analysis_results.sort_values(by='abs_rho', ascending=False).drop(columns=['abs_rho'])

print("--- KBO 정규시즌 강팀 구조 분석 결과 (2020-2025) ---")
print(sorted_results.to_string(index=False))

# # 4. 시각화 (상관계수 히트맵)
# plt.rcParams['font.family'] = 'Malgun Gothic' # 윈도우 한글 폰트 설정 (Mac은 AppleGothic)
# plt.figure(figsize=(10, 8))
# corr_df = df[[target] + features].rename(columns=feature_mapping).corr(method='spearman')
# sns.heatmap(corr_df, annot=True, cmap='coolwarm', fmt=".3f", linewidths=0.5)
# plt.title('KBO 주요 지표별 스피어먼 상관계수 히트맵')
# plt.show()

--- KBO 정규시즌 강팀 구조 분석 결과 (2020-2025) ---
       분석 지표  Spearman_Rho  P_value 통계적 유의성(0.05미만)
   WAR_total         0.906    0.000             유의함
 ERA (평균자책점)        -0.622    0.000             유의함
        WHIP        -0.580    0.000             유의함
      득점 (R)         0.559    0.000             유의함
OPS (출루+장타율)         0.510    0.000             유의함
    F% (수비율)         0.363    0.004             유의함
 SB% (도루성공률)        -0.027    0.838         유의하지 않음
